# 30 — Submission de prueba: texto combinado (variante a de la Fase T)

Modelo final (CatBoost P6b + HistGB P7a con duración, promedio 50/50) con `prob_texto_new` reemplazada por **(logística + BETO) / 2**. Es una submission **de prueba, sin subir**: usa las probabilidades de BETO guardadas en la Fase Q (no hay pesos), así que `predict.ipynb` todavía no la puede reproducir.

**Cómo se arma la columna de texto:**

| | Logística | BETO |
|---|---|---|
| Train (70.000 filas) | `PrediccionOOF.fit_transform`: OOF con la CV interna de 5 folds (semilla 42) | OOF de los 5 globales (`sin_k` predice el fold k) |
| Test (30.000 filas) | `PrediccionOOF.transform`: logística entrenada con todo el train | promedio de los 5 globales (`test_bert.csv`) |

Misma regla que en la CV: ninguna fila del train recibe la probabilidad de un modelo que la vio.

**Control:** con la logística sola como columna, tiene que salir **exactamente** `submission_p7c_duracion.csv`.

In [1]:
import sys
sys.path.append("..")
import os, time, warnings
import numpy as np
import pandas as pd

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      PrediccionOOF, armar_modelo_texto, armar_pipeline_histgb_combos, armar_pipeline_final, INDICES_CAT)

preparar = lambda df: agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(df))))
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
train = preparar(crudo)
test = preparar(pd.read_json("../data/test_data.jsonlines", lines=True))
n = len(train)
carpeta_lab = ".cache_laboratorio"

# Logística de texto: exactamente lo que hace PrediccionOOF adentro de los pipelines
columnas = ["title", "warranty_texto"]
inicio = time.time()
logistica = PrediccionOOF(armar_modelo_texto(), random_state=42)
L_train = logistica.fit_transform(train[columnas], y)[:, 0]
L_test = logistica.transform(test[columnas])[:, 0]
assert np.allclose(L_train, pd.read_csv(os.path.join(carpeta_lab, "oof_texto_logistica.csv"))["prob"])
print(f"Logística: {(time.time() - inicio) / 60:.1f} min (OOF del train = la de la CV)")

# BETO: OOF de los 5 globales en el train y su promedio en el test (Fase Q)
B_train = np.full(n, np.nan)
for k in range(1, 6):
    d = np.load(os.path.join(carpeta_lab, "bert", f"sin_{k}.npz"))
    es_train = d["filas"] < n
    B_train[d["filas"][es_train]] = d["prob"][es_train]
B_test = pd.read_csv(os.path.join(carpeta_lab, "test_bert.csv"))["bert_test_promedio"].to_numpy()
assert not np.isnan(B_train).any() and len(B_test) == len(test)
print("Correlación logística-BETO: train", np.corrcoef(L_train, B_train)[0, 1].round(3), "| test", np.corrcoef(L_test, B_test)[0, 1].round(3))

Logística: 0.5 min (OOF del train = la de la CV)
Correlación logística-BETO: train 0.889 | test 0.899


## Entrenamiento con todo el train y predicción

`armar(modelo)`: el pipeline de siempre, con el transformer `texto` reemplazado por `passthrough` de la columna `t_texto` (misma posición: `INDICES_CAT` no cambia). `submission(t_train, t_test)` entrena los dos modelos y devuelve P(new) promediada.

In [2]:
def armar(modelo):
    pipeline = armar_pipeline_histgb_combos() if modelo == "histgb" else armar_pipeline_final()
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [nm for nm, _, _ in transformers].index("texto")
    transformers[i] = ("texto", "passthrough", ["t_texto"])
    return pipeline


def prob_final(t_train, t_test):
    X, X_test = train.assign(t_texto=t_train), test.assign(t_texto=t_test)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        catboost = armar("catboost").fit(X, y, modelo__cat_features=INDICES_CAT)
        histgb = armar("histgb").fit(X, y)
    return 0.5 * catboost.predict_proba(X_test)[:, 1] + 0.5 * histgb.predict_proba(X_test)[:, 1]


def a_submission(prob):
    return pd.DataFrame({"ID": np.arange(1, len(test) + 1), "condition": np.where(prob >= 0.5, "new", "used")})


referencia = pd.read_csv("../submission/submission_p7c_duracion.csv")

inicio = time.time()
control = a_submission(prob_final(L_train, L_test))
print(f"Control ({(time.time() - inicio) / 60:.1f} min): predicciones distintas de submission_p7c_duracion.csv =",
      int((control["condition"] != referencia["condition"]).sum()))

Control (0.7 min): predicciones distintas de submission_p7c_duracion.csv = 0


In [3]:
inicio = time.time()
prob_a = prob_final((L_train + B_train) / 2, (L_test + B_test) / 2)
nueva = a_submission(prob_a)
print(f"Variante a: {(time.time() - inicio) / 60:.1f} min")

archivo = "../submission/submission_texto_combinado.csv"
nueva.to_csv(archivo, index=False)
leida = pd.read_csv(archivo)
assert leida.shape == (30000, 2)
assert list(leida.columns) == ["ID", "condition"]
assert (leida["ID"].to_numpy() == np.arange(1, 30001)).all()
assert leida["condition"].isin(["new", "used"]).all()

cambian = leida["condition"] != referencia["condition"]
print("Formato OK |", archivo)
print(f"Predicciones que cambian: {cambian.sum()} de 30.000 ({cambian.mean():.2%})")
print(pd.crosstab(referencia["condition"].rename("p7c_duracion"), leida["condition"].rename("texto_combinado")))
print("% new: antes", (referencia["condition"] == "new").mean().round(4), "| ahora", (leida["condition"] == "new").mean().round(4))
pd.DataFrame({"prob_new": prob_a}).to_csv(os.path.join(carpeta_lab, "test_texto_combinado.csv"), index=False)

Variante a: 0.7 min
Formato OK | ../submission/submission_texto_combinado.csv
Predicciones que cambian: 366 de 30.000 (1.22%)
texto_combinado    new   used
p7c_duracion                 
new              15680    187
used               179  13954
% new: antes 0.5289 | ahora 0.5286
